<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/basic_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/google-deepmind/main/examples/requirements.in' 'git+https://github.com/vishaljoshi24/concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
!pip install gdm-concordia[ollama]

Dependencies

In [ ]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
API_TYPE = 'ollama'
MODEL_NAME = 'llama3.2:1b'
API_KEY = ''
DISABLE_LANGUAGE_MODEL = False

In [ ]:
model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
    # api_key=API_KEY
)

In [ ]:
DISABLE_LANGUAGE_MODEL = False
if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [ ]:
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

D&D Agent Prefab

In [ ]:
import dataclasses
from concordia.typing import prefab as prefab_lib
from concordia.agents import entity_agent_with_logging
from concordia.components import agent as agent_components

@dataclasses.dataclass
class DnDAgent(prefab_lib.Prefab):
    description: str = "A custom agent that acts like a D&D player."

    def build(self, model, memory_bank):
        name = self.params.get("name", "Agent")
        goal = self.params.get("goal", "")
        backstory = self.params.get("backstory", "")
        profession = self.params.get("profession", "")

        # Create components
        memory = agent_components.memory.AssociativeMemory(
            memory_bank=memory_bank)
        instructions = agent_components.instructions.Instructions(
            agent_name=name)
        observation = agent_components.observation.LastNObservations(
            history_length=50)

        components = {
            agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: memory,
            "Instructions": instructions,
            agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: observation,
        }

        act_component = agent_components.concat_act_component.ConcatActComponent(
            model=model,
            component_order=list(components.keys()),
        )

        return entity_agent_with_logging.EntityAgentWithLogging(
            agent_name=name,
            act_component=act_component,
            context_components=components,
        )

# Register custom prefab
prefabs["my_custom__Entity"] = DnDAgent()

In [ ]:
prefabs['dndagent__Entity'] = DnDAgent()

In [ ]:
PLACE = 'Wizards Tower Brewing Company, Forgotten Realms'
CONTEXT = ('This D&D short campaign specifically concerns a craft brewery.',
'It is set in the Wizards Tower Brewing Company and is in dire need of help.',
'A band of reliable, affordable adventurers are needed to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
'For the duration of the one-shot, only Player_One, Player_Two and Brewery_Owner are in the brewery',
'At the beginning of this adventure Player_One and Player_Two',
'meet in the Wizard Tower Brewing Company. These two adventurers',
'DO NOT know each other AT FIRST and need to get to know each other.')

In [ ]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Player_One',
            'goal': 'To collaborate with Player_Two in completing the task given by Brewery_Owner'
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Player_Two',
            'goal': 'To collaborate with Player_One in completing the task given by Brewery_Owner'
        },
    ),
    # prefab_lib.InstanceConfig(
    #     prefab='dndagent__Entity',
    #     role=prefab_lib.Role.ENTITY,
    #     params={
    #         'name': Player_Three,
    #     },
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'acting_order': 'game_master_choice',
        },
    ),
]

In [ ]:
from concordia.environment.engines import sequential

engine = sequential.Sequential()

In [ ]:
config = prefab_lib.Config(
    default_premise=(
        CONTEXT
    ),
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

In [ ]:
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
    engine=engine,
)

In [ ]:
results_log = runnable_simulation.play(max_steps=1)